# ViFinQA — sinh payload masked-PAL có fusion + rerank (Kaggle T4)

Notebook này chạy **bước 1** của đường end-to-end: sinh payload ứng viên ô
cho LLM, dưới đúng cấu hình retrieval mà `submission export` sẽ dùng.

## Vì sao phải chạy GPU, và vì sao chỉ một lần

`ProgramDecision.cells` là **vị trí** trong danh sách ô ứng viên, mà danh
sách đó dựng từ `retrieved`. Nên `row-batches` và `export` bắt buộc xếp
hạng bảng bằng cùng một cấu hình — nghĩa là cross-encoder phải chấm đúng
1012 x 50 cặp đó **hai lần**.

`CachedReranker` ghi điểm ra đĩa, khoá theo (model đã pin, câu hỏi, danh
sách snippet). Lần thứ hai không còn gì để tính, và **không nạp model**:
cross-encoder chỉ được dựng khi cache miss.

| Bước | Chạy ở đâu | GPU |
|---|---|---|
| 1. `row-batches` (notebook này) | Kaggle T4 | có |
| 2. LLM sinh `ProgramDecision` | offline | tuỳ |
| 3. `export` (tải cache về) | máy bạn | **không** |

## Cache cũng chính là checkpoint

Cache ghi **một file mỗi câu hỏi**. Session Kaggle chết giữa chừng thì
chạy lại cell nặng chỉ tốn phần chưa xong — không mất giờ GPU đã bỏ ra.
Đó là lý do không cần cơ chế checkpoint riêng.

## Chuẩn bị

1. **Settings → Accelerator → GPU T4 ×2** — hai GPU, mỗi model một GPU:
   encoder dense chạy ở `cuda:1`, reranker ở `cuda:0`.
2. **Settings → Internet → On** (cần tải Qwen3-Reranker-4B).
3. Tạo **Kaggle Dataset** chứa, giữ nguyên cấu trúc:

```
vifinqa-artifacts/
  processed/release_v2_422df141c935/      # ~954MB
  indexes/bm25-v4/<fp>/  và  <fp>_row/    # ~1.8GB
  indexes/dense-qwen3-4b/<fp>/  và  corpus/   # ~1.6GB
  qa/week1_pilot_422df141c935/            # release lock + annotation
  interim/week1_gate/422df141c935/        # gate-result.json
  raw/ViFinQA/questions/questions.jsonl
```

Tổng ~4.4GB. Attach vào notebook.


## 1. Kiểm môi trường

Dừng ngay nếu thiếu GPU hoặc dung lượng.


In [ ]:
import shutil, subprocess

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
free_gb = shutil.disk_usage('/kaggle/working').free / 1e9
print(f'/kaggle/working free: {free_gb:.1f} GB')
assert free_gb > 12, 'can >12GB cho weights reranker + cache'

import torch
assert torch.cuda.is_available(), 'bat Settings -> Accelerator -> GPU'
print('GPU:', torch.cuda.get_device_name(0),
      f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


## 2. Tạo venv Python 3.11

Kernel Kaggle đang chạy Python 3.12, còn repo đòi `python <3.12`, nên cài
thẳng lên kernel sẽ fail. Dùng venv 3.11 và để `pyproject.toml` của repo
quyết định phiên bản phụ thuộc — không ghim tay từng gói (pin cũ
transformers 4.44 chưa hỗ trợ Qwen3).


In [ ]:
!apt-get -qq install -y python3.11 python3.11-venv
!python3.11 -m venv /kaggle/working/venv311
!/kaggle/working/venv311/bin/python --version


## 3. Lấy repo


In [ ]:
REPO = 'https://github.com/TranVu2005/financial-assistant'
BRANCH = 'main'

# %cd truoc luon o lai trong tien trinh kernel. Chay lai cell nay lan hai
# ma khong lui ra ngoai truoc se khien `rm -rf` xoa dung thu muc kernel
# dang dung trong do -- moi shell con sau do mat luon cwd (loi
# 'getcwd: cannot access parent directories'), du git clone da tao lai no.
%cd /kaggle/working
!rm -rf /kaggle/working/repo
!git clone -q --depth 1 --branch {BRANCH} {REPO} /kaggle/working/repo
%cd /kaggle/working/repo
# Cai non-editable tu ban clone bang pip cua venv: phu thuoc tu resolve
# theo pyproject, khong ghim tay (pin cu chua ho tro Qwen3).
!/kaggle/working/venv311/bin/pip install -q /kaggle/working/repo 2>&1 | tail -2
!/kaggle/working/venv311/bin/python -c "import financial_report_qa; print('pkg ok')"
!git log --oneline -1


## 4. Nối dữ liệu vào đúng chỗ repo mong đợi

Repo dùng đường dẫn tương đối (`data/processed/...`). Kaggle input là
read-only nên symlink vào, không copy — tiết kiệm 4.4GB và vài phút.

**Sửa `KAGGLE_DATA` cho khớp tên dataset của bạn.**


In [ ]:
import os
from pathlib import Path

KAGGLE_DATA = Path('/kaggle/input/vifinqa-artifacts')   # <-- SUA
assert KAGGLE_DATA.is_dir(), f'khong thay {KAGGLE_DATA}; kiem ten dataset da attach'

root = Path('/kaggle/working/repo')
(root / 'data').mkdir(exist_ok=True)
for name in ('processed', 'indexes', 'qa', 'interim', 'raw'):
    src = KAGGLE_DATA / name
    dst = root / 'data' / name
    if not src.is_dir():
        raise SystemExit(f'thieu {src} trong dataset')
    if dst.exists() and not dst.is_symlink():
        # data/qa la thu muc THAT sau git clone (gold files + release lock
        # duoc track trong git, khac processed/indexes/interim/raw bi
        # gitignore). Da co san tu clone -- khong symlink de khong xoa mat.
        print(f'{dst} da co san tu git clone, bo qua symlink')
        continue
    if dst.is_symlink():
        dst.unlink()
    os.symlink(src, dst, target_is_directory=True)
    print(f'{dst} -> {src}')

# data/qa/week1_pilot_.../dataset-pilot-v1.json phai co san tu git clone
lock = root / 'data/qa/week1_pilot_422df141c935/dataset-pilot-v1.json'
assert lock.is_file(), f'thieu {lock} -- git clone thieu file duoc track'
print('release lock (tu git clone):', lock)


## 5. Chốt chặn trước khi tốn giờ GPU

Ba thứ phải đúng, cả ba đều fail-closed. Cell này mất vài giây và bắt
được mọi lỗi ghép dữ liệu trước khi bạn chạy cell 40 phút.

Các cell chạy lệnh trong notebook này đều tự định nghĩa đủ biến đường
dẫn của chính nó, nên sau khi restart kernel chỉ cần chạy lại đúng cell
cần dùng — không có biến nào bị quên thành chuỗi `{LOCK}` nguyên văn.


In [ ]:
%%bash
/kaggle/working/venv311/bin/python - <<'PY'
import json
from pathlib import Path

from financial_report_qa.retrieval.release import resolve_retrieval_release

FP = '422df141c935d46bfd14302abec50f32380e6e4c012159f8ad0ae5560c8a446a'
LOCK = Path('data/qa/week1_pilot_422df141c935/dataset-pilot-v1.json')
BM25 = Path(f'data/indexes/bm25-v4/{FP}')
DENSE = Path(f'data/indexes/dense-qwen3-4b/{FP}')

# 1) release lock: fingerprint, gate result, so dong parquet that
release = resolve_retrieval_release(LOCK, repo_root=Path.cwd())
print('release OK   :', release.dataset_fingerprint[:16], '...')

# 2) dense corpus phai khai dung lock da sinh ra no
corpus_manifest = json.loads((DENSE.parent / 'corpus' / 'manifest.json').read_text())
assert corpus_manifest['release_lock_sha256'] == release.lock_sha256, \
    'dense corpus dung tu mot release lock KHAC -- khong dung duoc'
print('dense corpus :', corpus_manifest['document_count'], 'docs')

# 3) row index cho row-fusion
row_index = Path(f'{BM25}_row')
assert row_index.is_dir(), f'thieu {row_index} -- row fusion se khong chay'
print('row index OK :', row_index.name)

questions = Path('data/raw/ViFinQA/questions/questions.jsonl')
print('questions    :', sum(1 for x in questions.open(encoding='utf-8') if x.strip()))
PY


## 6. Tải trước reranker

Model ~8GB (checkpoint gốc là bf16). Nạp thử ở bfloat16 lên `cuda:0` — đúng
cấu hình của cell nặng, nơi reranker sống ở `cuda:0` còn encoder dense ở
`cuda:1` — để vừa VRAM T4, và tách riêng để một lỗi mạng HuggingFace không
làm bạn tưởng pipeline hỏng.


In [ ]:
%%bash
/kaggle/working/venv311/bin/python - <<'PY'
import gc

import torch

from financial_report_qa.retrieval.reranker import (
    Qwen3CrossEncoderReranker,
    approved_reranker_spec,
)

spec = approved_reranker_spec('qwen3-reranker-4b')
print('model :', spec.model_id)
print('pinned:', spec.revision)
# Diem rerank h nay tinh theo duong CHINH TAC causal yes/no cua model card
# Qwen3-Reranker: logit('yes') - logit('no') tai vi tri cuoi qua judge chat
# template. Truoc do code nap qua AutoModelForSequenceClassification, trong
# khi checkpoint chi chua trong luong Qwen3ForCausalLM (khong co head phan
# loai) -- transformers LOAD REPORT bao 'score.weight | MISSING | newly
# initialized' va moi diem chi la phep chieu ngau nhien.

_warm = Qwen3CrossEncoderReranker(spec, model_dtype='bfloat16', device='cuda:0')
print('reranker nap xong (bf16, cuda:0)')
del _warm
gc.collect()
torch.cuda.empty_cache()
PY


## 7. Sinh payload — cell nặng

Chỗ tốn giờ GPU: 1012 câu x 50 ứng viên.

**Năm cờ retrieval dưới đây phải trùng y hệt khi chạy `export`.** Phần lớn
nằm trong sidecar `retrieval-fingerprint.json`, mà
`export --assert-payload-fingerprint` từ chối chạy nếu lệch; riêng
`--rerank-dtype` không nằm trong sidecar — tự bảo đảm truyền đúng (bf16 lúc
sinh payload thì bf16 lúc export): lệch precision làm các điểm gần ngang
nhau có thể xê dịch hạng.

`--rerank-dtype` chỉ hạ precision TÍNH TOÁN cho vừa VRAM T4 (fp32 thì OOM);
điểm chấm ghi ra cache vẫn float32.

Ba cờ đặt chỗ GPU — `--table-encoder-device cuda:1`,
`--table-encoder-model-dtype float16`, `--rerank-device cuda:0` — cùng tiền
lệ compute-only: không thuộc spec, không nằm trong sidecar. Chúng chỉ quyết
định model nào sống ở GPU nào trên máy hai T4 này (encoder một GPU,
reranker GPU kia; fp32 4B không thể vừa một T4 nên encoder cuda* mặc định
float16). Export chạy CPU nên KHÔNG truyền chúng; lệch precision embed
giữa hai lần chạy chỉ nằm trong giới hạn làm tròn fp16 — cùng lớp nhiễu
đã chấp nhận ở `--rerank-dtype`.

Session chết giữa chừng: chạy lại **chính cell này**. Cache giữ mọi câu
đã chấm, lần chạy lại chỉ tốn phần còn thiếu.

Cell này tự chứa đủ biến của nó (không đọc biến từ cell trước), nên việc
chạy lại sau khi restart kernel luôn an toàn.


In [ ]:
import os

# Cell tu chua du bien cua no: sau khi restart kernel chi can chay lai
# dung cell nay, khong phu thuoc bien tu cell truoc (thieu bien thi
# argparse nhan chu '{LOCK}' nguyen van va fail bam).
FP = '422df141c935d46bfd14302abec50f32380e6e4c012159f8ad0ae5560c8a446a'
LOCK = 'data/qa/week1_pilot_422df141c935/dataset-pilot-v1.json'
BM25 = f'data/indexes/bm25-v4/{FP}'
DENSE = f'data/indexes/dense-qwen3-4b/{FP}'
OUT = 'artifacts/batches/program-full'
CACHE = '/kaggle/working/rerank-score-cache'

os.chdir('/kaggle/working/repo')

!mkdir -p {CACHE}
# Hai GPU, moi model mot GPU (compute/placement-only, khong thuoc spec):
# encoder dense o cuda:1 tinh fp16 de vua VRAM T4, reranker o cuda:0.
!PYTHONIOENCODING=utf-8 /kaggle/working/venv311/bin/python -m financial_report_qa.cli submission row-batches --release-lock {LOCK} --bm25-index {BM25} --questions-path data/raw/ViFinQA/questions/questions.jsonl --release-dir data/processed/release_v2_422df141c935 --output-dir {OUT} --dense-index {DENSE} --table-dense-weight 1.0 --rerank --rerank-dtype bfloat16 --rerank-cache-dir {CACHE} --k 10 --table-encoder-device cuda:1 --table-encoder-model-dtype float16 --rerank-device cuda:0


## 8. Kiểm kết quả


In [ ]:
import json
import os
from pathlib import Path

# Tu chua du bien cua minh (an toan sau khi restart kernel).
os.chdir('/kaggle/working/repo')
OUT = 'artifacts/batches/program-full'
CACHE = '/kaggle/working/rerank-score-cache'

out = Path(OUT)
batches = sorted(out.glob('batch_*.jsonl'))
total = sum(sum(1 for x in b.open(encoding='utf-8') if x.strip()) for b in batches)
print(f'{len(batches)} file batch, {total} cau')

sidecar = json.loads((out / 'retrieval-fingerprint.json').read_text(encoding='utf-8'))
print(json.dumps(sidecar, indent=2, ensure_ascii=False))
assert sidecar['reranker_enabled'] is True, 'rerank KHONG chay -- kiem lai co'
assert sidecar['dense_index'] is not None, 'dense KHONG bat -- kiem lai co'

cached = list(Path(CACHE).rglob('*.npy'))
size_mb = sum(p.stat().st_size for p in cached) / 1e6
print(f'cache: {len(cached)} cau da cham, {size_mb:.1f} MB')

first = json.loads(batches[0].read_text(encoding='utf-8').splitlines()[0])
print('\ncau mau:', first['question'][:90])
print('so ung vien o:', len(first['candidates']))
print('ung vien[0]:', json.dumps(first['candidates'][0], ensure_ascii=False))
assert 'value' not in json.dumps(first), 'payload lo gia tri so -- vi pham N7'
print('\nN7 OK: payload khong mang mot gia tri so nao')


## 9. Đóng gói mang về


In [ ]:
!cd /kaggle/working && tar czf vifinqa-batches.tar.gz -C /kaggle/working/repo artifacts/batches/program-full -C /kaggle/working rerank-score-cache
!ls -lh /kaggle/working/vifinqa-batches.tar.gz
print('Tai file nay ve, giai nen vao goc repo o may ban.')


## 10. Việc còn lại (không chạy ở đây)

### Bước 2 — LLM sinh quyết định (offline)

Đọc `artifacts/batches/program-full/batch_*.jsonl`, mỗi câu trả một dòng
JSON đúng schema `ProgramDecision`:

```json
{"question_id": 17, "cells": [42, 17],
 "program": "([NUM_0] - [NUM_1]) / [NUM_1]",
 "uses": [{"num": 0, "row": "Doanh thu thuần", "col": "Năm 2023"},
           {"num": 1, "row": "Doanh thu thuần", "col": "Năm 2022"}],
 "scale": "percent"}
```

`program` **không được chứa một literal số nào** — linter C8 parse AST và
fail build nếu có. Đổi thang bằng `scale`, không bằng `* 100`.

Kiểm file trước khi dùng:

```bash
uv run python -c "from pathlib import Path; from financial_report_qa.planning.program_decisions import load_program_decisions; print(len(load_program_decisions(Path('data/decisions/program-full.jsonl'))), 'decisions')"
```

### Bước 3 — export (máy bạn, KHÔNG cần GPU)

Giải nén `rerank-score-cache` vào `data/indexes/rerank-score-cache/`, rồi:

```bash
uv run financial-report-qa submission export --release-lock data/qa/week1_pilot_422df141c935/dataset-pilot-v1.json --bm25-index data/indexes/bm25-v4/422df141c935d46bfd14302abec50f32380e6e4c012159f8ad0ae5560c8a446a --questions-path data/raw/ViFinQA/questions/questions.jsonl --execution-config configs/base.yaml --program-decisions data/decisions/program-full.jsonl --dense-index data/indexes/dense-qwen3-4b/422df141c935d46bfd14302abec50f32380e6e4c012159f8ad0ae5560c8a446a --table-dense-weight 1.0 --rerank --rerank-dtype bfloat16 --rerank-cache-dir data/indexes/rerank-score-cache --assert-payload-fingerprint artifacts/batches/program-full/retrieval-fingerprint.json --output-zip artifacts/submissions/dense-rerank-k10.zip --report-dir artifacts/reports/dense-rerank-k10 --k 10
```

Năm cờ retrieval trùng notebook này (gồm `--rerank-dtype bfloat16`). **Luôn truyền**
`--assert-payload-fingerprint` — thiếu nó, một cờ lệch sẽ dịch mọi chỉ số
`cells` trong im lặng thay vì fail ngay.

Ba cờ đặt chỗ GPU của notebook (`--table-encoder-device`,
`--table-encoder-model-dtype`, `--rerank-device`) là knob compute-only cho
máy hai T4 — KHÔNG truyền ở export: mặc định CPU/fp32 giữ đúng hành vi cũ.

Encoder dense vẫn nạp lúc export để embed từng câu hỏi (~16GB RAM), nhưng
reranker thì không: cache đầy nên `CachedReranker` không dựng model.

> `RerankModelError: rerank cache miss` khi export nghĩa là cache thiếu câu
> nào đó — gần như luôn do một cờ retrieval lệch làm danh sách ứng viên
> đổi. Đối chiếu lại sidecar, đừng chạy tiếp không rerank.
